# FACE resistant depression cohort: a first tutorial

This notebook shows how to load the FACE DR data (treatment-resistant depression) with the `facedata` package and how to take a first look at them. It is written for people who have never used this cohort and who do not code much.

You will learn how to:

1. load the data and understand the six tables you get;
2. find a variable and read what it means;
3. look at rating scales, lab results, neuropsychology, treatments and history at the inclusion visit;
4. follow a few patients over time, and the whole cohort over time;
5. judge how much a value can be trusted, and avoid the traps of this cohort;
6. work through four questions: **response and remission**, **resistance and treatments**, **MINI comorbidities**, **life events and trauma**;
7. build your own table and save it;
8. **put the bipolar, schizophrenia and depression cohorts in one table.**

**How to use a notebook.** A notebook is made of cells. Click in a grey cell and press `Shift` + `Enter` to run it: the result appears under it. Run the cells from top to bottom, because each one uses what the cells above it made. You can change a cell and run it again as often as you like; nothing you do here changes the data file.

## Before you start: these are patient data

The results that appear under the cells are patient data, even if the patients have no name.

- **Before you share this notebook or send it to GitHub, clear the results:** menu *Edit*, then *Clear Outputs of All Cells*, then save.
- Never copy patient rows into an e-mail, a chat or a slide. Share counts, means and figures of groups.
- Save your own tables in the folder `data/exports/`. Everything in `data/` stays on this computer.

## 1. Load the cohort

The first cell brings in the tools. The second one reads the extraction: it takes about half a minute, so wait for the number in brackets on the left of the cell to appear.

If your extraction file has another name, change it in the first cell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from facedata import dr

EXTRACTION = "data/dr/DRALL_20261030.tsv"          # the file you received
pd.set_option("display.max_columns", 30)

In [ ]:
result = dr.load(EXTRACTION)

`result` holds six tables. You will mostly use the first two.

| Table | One row per | What it holds |
|---|---|---|
| `result.data` | patient visit | everything measured at a visit: scales, lab results, tests, answers |
| `result.patients` | patient | what does not change: sex, date of birth, the parents and grandparents |
| `result.treatments` | patient and medication | the medications, in three lists: current psychotropic, past psychotropic, somatic |
| `result.relatives` | patient and relative | the family history, relative by relative |
| `result.qc` | variable | the catalogue: every variable, its label and how well it is filled |
| `result.flags` | flagged cell | every value that was repaired, doubted or removed, with the reason |

Let us give the first two a short name and count.

In [ ]:
data = result.data
patients = result.patients

print(len(data), "rows of", data["usubjid"].nunique(), "patients")
print(data.shape[1], "columns in data and", patients.shape[1], "in patients")

## 2. What a visit looks like

Every row of `data` starts with the same columns:

- `cohort`: `dr` on every row. It matters when you put several cohorts together (section 19).
- `usubjid`: the patient's number. It links all the tables.
- `visit`: the name of the visit. `screening` comes first, `V0` is the inclusion visit, then `V6_mois` (six months), `V1_an` (one year), `V18_mois`, `V2_ans` and `V4_ans`.
- `visit_month`: the planned number of months since inclusion (0, 6, 12, 18, 24, 48). Easier to plot than the names.
- `visit_date`, `siteid` (the centre), `age` (at the visit).

The cell below shows the first rows, with two scores next to these columns.

In [ ]:
data[["cohort", "usubjid", "visit", "visit_month", "visit_date", "siteid", "age", "madrs_total", "cgi_severity"]].head()

How many visits of each kind are there? `value_counts()` counts the rows for each value, and `.plot.bar()` draws the counts.

In [ ]:
data["visit"].value_counts().plot.bar(figsize=(10, 3), title="Number of visits of each kind")
plt.show()

Most analyses start from the inclusion visit. The next cell keeps only the rows where `visit` is `V0`, and counts the inclusions per year.

**The pattern to remember:** `table[table["column"] == value]` keeps the rows where the column has that value.

In [ ]:
inclusion = data[data["visit"] == "V0"]

inclusion["visit_date"].dt.year.value_counts().sort_index().plot.bar(figsize=(10, 3), title="Inclusion visits per year")
plt.show()

**The screening visit is a real visit in this cohort.** Every patient has one, about ten days before the inclusion. It is where the diagnostic interview (MINI), the stage of resistance (Thase and Rush) and the inclusion criteria are recorded, with a first MADRS and a first QIDS. These answers are on the `screening` row, not on the `V0` row.

The cell below counts, for five variables, how many screening rows and how many inclusion rows hold a value. Section 15 shows how to put the two rows of a patient side by side.

In [ ]:
screening = data[data["visit"] == "screening"]
look_at = ["madrs_total", "qids_total", "thase_rush_stage", "mini_mde_criteria", "eligible"]

pd.DataFrame({"screening rows": screening[look_at].notna().sum(), "inclusion rows": inclusion[look_at].notna().sum()})

## 3. Who the patients are

Sex, diagnosis and age at inclusion are in `patients`, one row per patient.

In [ ]:
print(patients["sex"].value_counts())
print()
print(patients["diagnosis"].value_counts())

patients["age_at_inclusion"].plot.hist(bins=30, title="Age at inclusion (years)")
plt.show()

To use these columns together with a visit, join the two tables on the patient's number with `merge`. From now on `inclusion` also has the sex.

The course of the depression is asked at the inclusion visit: a single episode or a recurrent disorder (`mdd_course`), and the number of depressive episodes in the life (`dsm4_lt_n_mde`).

In [ ]:
inclusion = inclusion.merge(patients[["usubjid", "sex", "ever_ect", "ever_tms", "ever_psychotherapy"]], on="usubjid")

print(inclusion.groupby("sex")["age"].agg(["count", "mean"]).round(1))
print()
print(inclusion["mdd_course"].value_counts())
print()
print(inclusion["dsm4_lt_n_mde"].describe().round(1))

## 4. Find a variable

There are about two thousand five hundred variables. `result.qc` is their catalogue: the name to type, a label, the instrument it belongs to, the dictionary tab, and `n_valid`, the number of values that can be used.

The cell below searches the labels for a word. Change `"sleep"` to anything: `"suicide"`, `"lithium"`, `"cholesterol"`...

In [ ]:
catalogue = result.qc[["variable", "label", "instrument", "tab", "type", "unit", "level", "n_valid"]]

catalogue[catalogue["label"].str.contains("sleep", case=False)]

The column `level` tells you in which table the variable is: `visit` means `data`, `patient` means `patients`, `list` means `treatments` and `relatives` means `relatives`.

You can also list everything one instrument holds, or count the instruments of each tab.

In [ ]:
print(catalogue.groupby("tab")["instrument"].nunique())

catalogue[catalogue["instrument"] == "madrs"]

**Read the notes of an instrument before you use it.** Each instrument has a short text that says what was asked, how the score is computed and what to be careful about. This is where the traps are written. For the childhood trauma questionnaire (CTQ), the notes say why the scores stored in the extraction are wrong and how they were corrected.

In [ ]:
from facedata.schema import load_schemas

notes = load_schemas(dr.SCHEMAS)
print(notes["ctq"]["notes"])

## 5. Rating scales and questionnaires

Five scores at the inclusion visit: the MADRS (depression, rated by the clinician, 0 to 60), the QIDS-SR16 (depression, rated by the patient, 0 to 27), the CGI severity (1 to 7), the GAF (functioning, 0 to 100, higher is better) and the SHAPS (anhedonia, 0 to 14).

`describe()` gives the number of values, the mean, the spread and the quartiles.

In [ ]:
scores = ["madrs_total", "qids_total", "cgi_severity", "gaf", "shaps_total"]

inclusion[scores].describe().round(1)

A histogram shows how the values are spread. These scores are whole numbers, so a few bars look too low or too high only because of where the bars are cut: change `bins=30` to another number and run the cell again to see.

In [ ]:
inclusion[scores].hist(bins=30, figsize=(10, 6))
plt.show()

Do the clinician and the patient agree on the severity? Each dot is one patient at inclusion.

In [ ]:
inclusion.plot.scatter(x="madrs_total", y="qids_total", alpha=0.3, title="Depression rated by the clinician (MADRS) and by the patient (QIDS)")
plt.show()

inclusion[["madrs_total", "qids_total"]].corr().round(2)

Compare groups with `groupby`: the mean of each score for men and women, then a box plot of the MADRS.

In [ ]:
print(inclusion.groupby("sex")[scores].mean().round(1))

inclusion.boxplot(column="madrs_total", by="sex")
plt.show()

**Three things to know about these scales.**

- **Four scores were wrong in the database and are delivered corrected:** the childhood trauma questionnaire (`ctq_total` and its two neglect scores), the impulsivity scale (`bis10_total`), the adherence scale (`mars_total`) and the anhedonia scale (`shaps_total`). The export computes the first three with the key of the bipolar cohort, and the SHAPS with five items turned that the form does not turn; the forms themselves are right. If you have the raw extraction, do not use its stored scores for these scales.
- **The QIDS total is computed by the package.** Since 2022 the form asks for one item of each appetite and weight pair, and the export, which wants both, stores no total on those visits (227 visits); the package computes it from the answers.
- **A set of scales is half empty since 2018**: the STAI, the Altman scale, the MAThyS, the PSQI and about ten others. The cell below shows it for the STAI, next to the MADRS, which did not move.

The other questionnaires work the same way: `staya_total` (anxiety), `mars_total` (adherence), `ctq_total` (childhood trauma), `psqi_total` (sleep), `fast_total` (functioning), `eq5d_vas` (health today, 0 to 100).

In [ ]:
year = inclusion["visit_date"].dt.year

(inclusion[["madrs_total", "staya_total"]].notna().groupby(year).mean() * 100).plot(marker="o", figsize=(10, 3.5), title="Inclusion visits that hold the score (%)")
plt.show()

inclusion[["staya_total", "mars_total", "ctq_total", "psqi_total", "fast_total", "eq5d_vas"]].describe().round(1)

## 6. Lab results

Lab results arrive from many laboratories, in several units. The package brings each one to a single unit, shown in the catalogue.

In [ ]:
labs = ["glucose", "crp", "tsh", "vitamin_d", "lithium"]

catalogue[catalogue["variable"].isin(labs)]

In [ ]:
inclusion[labs].describe().round(2)

**Results below the detection limit.** A laboratory sometimes answers "less than 5" instead of a number. The value then holds the limit (5), and a second column ending in `_cens` says so: `-1` means "the true value is below", `0` means measured, `1` means "above".

In [ ]:
inclusion["crp_cens"].value_counts()

Two pictures: vitamin D at inclusion, and the lithium level of the visits that have one. The lithium level is measured only for the patients who take lithium, so it is filled at few visits.

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.5))

inclusion["vitamin_d"].plot.hist(bins=30, ax=left, title="Vitamin D at inclusion")
data["lithium"].plot.hist(bins=30, ax=right, title="Lithium level, all visits")
plt.show()

Body mass index, cut into the usual classes with `pd.cut`.

In [ ]:
bmi_class = pd.cut(inclusion["bmi"], [0, 18.5, 25, 30, 100], labels=["under 18.5", "18.5 to 25", "25 to 30", "30 and more"])

bmi_class.value_counts(sort=False).plot.bar(title="Body mass index at inclusion")
plt.show()

## 7. Neuropsychology

The assessment is done at inclusion and at one, two and four years; never at six or eighteen months.

Many results come as a **z-score**: 0 is the average of people of the same age in the general population, and here higher is always better. A z-score of -1 is one standard deviation below that average.

In [ ]:
print(data[data["tmt_a_time"].notna()]["visit"].value_counts())

tests = ["tmt_a_time_z", "tmt_b_time_z", "wais4_ds_z", "wais4_simil_z", "wais4_code_z", "rlri16_free1_z"]

inclusion[tests].plot.box(figsize=(11, 4), title="Z-scores at inclusion (0 is the population average)")
plt.axhline(0, color="grey")
plt.show()

**Two forms of the same test are never mixed.** The verbal fluency has two forms, with other letters and categories; the second one is mostly used at the one-year visit, so that a patient does not get the same words twice. Each form has its own columns (`fluency_...` and `fluency2_...`), and `fluency_form` says which one a visit used.

In [ ]:
print(pd.crosstab(data["visit"], data["fluency_form"]))

data.groupby("fluency_form")[["fluency_p_words", "fluency2_letter_words"]].agg(["count", "mean"]).round(1)

**Fewer assessments, and three tests retired.** The cell below counts, year by year, the share of inclusion visits that hold three tests. The memory for designs is no longer done since 2021.

In [ ]:
(inclusion[["tmt_a_time", "auditory_span_forward", "designs1_raw"]].notna().groupby(year).mean() * 100).plot(marker="o", figsize=(10, 3.5), title="Inclusion visits that hold the test (%)")
plt.show()

## 8. Treatments

`result.treatments` has one row per patient and medication. The column `list` says which of the three lists the row comes from:

- `medications`: the psychotropic treatments recorded as current;
- `medications_lifetime`: a psychotropic treatment of the past;
- `medications_somatic`: the somatic treatments.

In [ ]:
treatments = result.treatments
print(treatments["list"].value_counts())

current = treatments[treatments["list"] == "medications"]
print()
print(len(current), "current psychotropic treatments of", current["usubjid"].nunique(), "patients")

figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
current["treatment_group"].value_counts().sort_values().plot.barh(ax=left, title="Current treatments per group")
current["molecule"].value_counts().head(15).sort_values().plot.barh(ax=right, title="The 15 most frequent molecules")
plt.tight_layout()
plt.show()

**The lists of this extraction are damaged.** For a patient with several products, the export does not say which dose and which dates belong to which product. For these patients the table gives one row per product, with no dose and no date. For a patient with one product, the doses and dates are right.

The columns of `data` that start with `on_` say whether a patient was on a group of treatments on the day of the visit (1 yes, 0 no). They need the dates, so they are empty for most patients with several products. Always look at `count` before you use them. Section 14 shows what can be done without the dates.

In [ ]:
groups = ["on_antidepressant", "on_lithium", "on_antipsychotic_2g", "on_anticonvulsant", "on_benzodiazepine"]

table = inclusion[groups].agg(["count", "mean"]).T
table["percent"] = (table["mean"] * 100).round(0)
print(len(inclusion), "inclusion visits")
table[["count", "percent"]]

## 9. History, comorbidities and family

Yes/no answers are stored as 1 and 0, so the mean of a column is the share of "yes". Always show the number of answers next to it.

These columns end in `_v0` or `_lifetime`: they are the answers of the inclusion visit, about the whole life (see the traps in section 12).

In [ ]:
answers = ["hypertension_v0", "diabetes_v0", "migraine_v0", "thyroid_disorder_v0", "cssrs_attempt_lifetime", "ever_ect", "ever_psychotherapy"]

table = inclusion[answers].agg(["count", "mean"]).T
table["percent"] = (table["mean"] * 100).round(1)
table[["count", "percent"]]

The history of the mood disorder is asked at inclusion too: the age at the first episode and at the first psychotropic treatment.

In [ ]:
print(inclusion[["dsm4_first_episode_age", "dsm4_lt_first_treatment_age", "dsm4_lt_n_mde"]].describe().round(1))

inclusion["dsm4_first_episode_age"].plot.hist(bins=30, title="Age at the first depressive episode (years)")
plt.show()

The family history comes in two forms. The table `relatives` has one row per relative (the brothers and sisters share one row, and so do the children). The table `patients` holds, for each parent and grandparent, whether a psychiatric history is known.

In [ ]:
relatives = result.relatives

print(relatives.groupby("kinship")["relative_mood_disorder"].agg(["count", "mean"]).round(2))
print()
print(patients["mother_psychiatric_history"].value_counts(dropna=False))

## 10. Follow patients over time

### A few patients

We look for patients who have a MADRS at five visits or more, and take six of them. `random_state=1` makes it the same six every time; change the number to see others.

The patients are called "Patient 1" to "Patient 6" in the figure, so that no patient number is printed.

In [ ]:
followed = data[data["madrs_total"].notna() & data["visit_month"].notna() & (data["visit"] != "screening")]

visits_per_patient = followed.groupby("usubjid").size()
often_seen = visits_per_patient[visits_per_patient >= 5].index.to_series()
chosen = often_seen.sample(6, random_state=1).tolist()

print(len(often_seen), "patients have a MADRS at five visits or more")

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for number, patient in enumerate(chosen, start=1):
    visits = followed[followed["usubjid"] == patient].sort_values("visit_month")
    left.plot(visits["visit_month"], visits["madrs_total"], marker="o", label=f"Patient {number}")
    right.plot(visits["visit_month"], visits["qids_total"], marker="o")

left.set_title("MADRS (clinician)")
right.set_title("QIDS (patient)")
left.set_xlabel("Months since inclusion")
right.set_xlabel("Months since inclusion")
left.legend()
plt.show()

The whole follow-up of one patient as a table: choose the columns you want to read side by side.

In [ ]:
one = followed[followed["usubjid"] == chosen[0]].sort_values("visit_month")

one[["visit", "visit_date", "age", "madrs_total", "qids_total", "cgi_severity", "gaf", "bmi"]]

### The whole cohort

The mean score at inclusion and at each follow-up visit. Look at `count` first: fewer and fewer patients are seen.

In [ ]:
kinds = ["V0", "V6_mois", "V1_an", "V18_mois", "V2_ans", "V4_ans"]
planned = data[data["visit"].isin(kinds)]

summary = planned.groupby("visit_month")[["madrs_total", "qids_total"]].agg(["count", "mean"]).round(1)
print(summary)

planned.groupby("visit_month")[["madrs_total", "qids_total"]].mean().plot(marker="o", title="Mean score at each visit")
plt.xlabel("Months since inclusion")
plt.show()

**Be careful with this figure.** The patients seen at one year are not all the patients included: some left, some were included too recently. If those who stay were less ill from the start, the mean goes down even if nobody improves. The cell below compares, at inclusion, the patients who were seen at one year with the others.

Section 13 looks at change the fair way: each patient compared with themself.

In [ ]:
seen_at_1_year = data[data["visit"] == "V1_an"]["usubjid"]
inclusion["seen_at_1_year"] = inclusion["usubjid"].isin(seen_at_1_year)

inclusion.groupby("seen_at_1_year")[["age", "madrs_total", "qids_total", "dsm4_lt_n_mde"]].agg(["count", "mean"]).round(1)

Cognition is measured at inclusion and at one, two and four years. `pivot` puts two visits side by side, one row per patient, and `dropna()` keeps the patients who have both. Here: the time to complete part B of the Trail Making Test, in seconds (less is better).

In [ ]:
tmt = data[data["visit"].isin(["V0", "V1_an"])].pivot(index="usubjid", columns="visit", values="tmt_b_time").dropna()

print(len(tmt), "patients have a TMT B at inclusion and at one year")
(tmt["V1_an"] - tmt["V0"]).plot.hist(bins=30, title="Change of the TMT B time between inclusion and one year (seconds)")
plt.show()

## 11. How much can a value be trusted?

While loading, the package repairs what it can and writes down everything it does in `result.flags`. There are three levels:

- `info`: the value was repaired or annotated and can be trusted (a unit converted, a "less than 5", a code for "does not know").
- `discrepancy`: the value is kept, but something disagrees with it. Decide for yourself.
- `danger`: the value could not be used. It is empty in `data`, and the text that was typed is kept in the flags.

In [ ]:
flags = result.flags
print(flags["severity"].value_counts())
print()
print(flags[flags["variable"] == "vitamin_b12"]["flag"].value_counts())

For vitamin B12, `unit_converted` means that results written in another unit were brought to the unit of the catalogue; `unit_unknown` means that no unit could be given, so the result is empty. Each flag keeps the text as it was typed (`raw`) and the value now delivered (`value`).

The catalogue also gives, for each variable, how many cells carry each kind of flag.

In [ ]:
result.qc[result.qc["variable"].isin(["vitamin_b12", "crp", "madrs_total", "mini_mde_current"])][["variable", "n_filled", "n_valid", "n_info", "n_discrepancy", "n_danger", "min", "median", "max"]]

**An example of a doubt: a box that the form fills by itself.** In the MINI, the box "the depressive episode is current" (`mini_mde_current`) holds 0 as soon as the form is opened, also when nobody answered the question. So a 0 can mean "no" or "not asked". The package keeps the box and puts the flag `derived_doubt` where the module has no answer.

The cell below counts the symptoms of the last two weeks (`mini_mde_symptoms_2_weeks`, five or more are needed for an episode) against the box. Many visits have five symptoms or more and a box at 0.

In [ ]:
five_symptoms = (data["mini_mde_symptoms_2_weeks"] >= 5).where(data["mini_mde_symptoms_2_weeks"].notna())

print(pd.crosstab(five_symptoms, data["mini_mde_current"], rownames=["five symptoms or more"], colnames=["box 'current'"]))
print()
print(flags[flags["flag"] == "derived_doubt"]["variable"].value_counts().head(8))

**If you want to be strict**, you can empty the values that carry a `discrepancy` flag before an analysis. The cell below does it for this box, in a copy of the table.

In [ ]:
doubted = flags[(flags["variable"] == "mini_mde_current") & (flags["severity"] == "discrepancy")][["usubjid", "visit"]].drop_duplicates()
doubted["doubted"] = True

strict = data.merge(doubted, on=["usubjid", "visit"], how="left")
strict.loc[strict["doubted"] == True, "mini_mde_current"] = float("nan")

print("Boxes:", data["mini_mde_current"].notna().sum(), "before,", strict["mini_mde_current"].notna().sum(), "after")

**The quality-control report.** The package can also write a small website with one page per instrument: the notes, the quality figures and four pictures per variable (by year, by centre). It takes several minutes. Remove the `#` in the cell below to make it, then open `reports/dr/index.html` in a web browser.

In [ ]:
# dr.report(result, "reports/dr")

## 12. The traps of this cohort

Read this list once; it will save you from the most common mistakes.

1. **The screening visit is a row of its own.** The MINI, the Thase and Rush stage and the inclusion criteria are on the `screening` row of a patient, not on the `V0` row. Section 15 shows how to bring them together.
2. **`_lifetime` and `_since_visit` are two different questions.** For the suicide scales, the mood history, the electroconvulsive therapy, the psychotherapies, the hospitalisations and the medical history, the inclusion form asks about the whole life (`_lifetime`, `_v0`, `ever_...`) and the follow-up form about the time since the last visit (`_since_visit`, `_fu`). Never put the two in one column.
3. **Empty does not mean "no".** An empty cell means not asked, not answered or not usable. A "no" is written 0.
4. **But a 0 in a box can mean "not asked".** In the MINI, the family form and the traumatic events, some boxes hold 0 on every row where the form was opened (section 11). Use the symptoms and the questions, and look at the flag `derived_doubt`.
5. **Sex and the parents are in `patients`.** Join them to the visits with `merge`, as in section 3.
6. **Patients leave.** Half are seen at six months, fewer than half at one year, a quarter at two years. Every figure over time mixes change and departures (sections 10 and 13).
7. **The six-month and eighteen-month visits are short.** They hold the depression scales, the weight and the laboratory tests, but no neuropsychology, no MINI and no history.
8. **A set of scales is half empty since 2018**: eleven self-questionnaires (STAI, Altman, MAThyS, Rosenberg, PSQI, Epworth, CSM, MARS, PRISE-M, BIS-10, BFI) and three clinician scales (YMRS, FAST, BAS). Most of it is the simplified visit of the application, which leaves these forms out (an empty scale is then "not asked", not "no"); the extraction does not say which visits are simplified, and some centres use it much more than others. The cell below shows it per centre.
9. **Centres differ.** They do not include the same patients and do not fill the forms the same way. Look at your variable per centre and per year before you conclude.
10. **Some scores were wrong in the database.** The CTQ, the BIS-10, the MARS, the SHAPS and the Fagerström total are delivered corrected, and the QIDS total is computed where the database stores none.
11. **The medication lists are damaged** (section 8): for a patient with several products there is no dose and no date, and the `on_...` columns are often empty.
12. **Extreme values of measures are brought back** to the 1st and 99th percentile (flag `winsorised`). The typed value is kept in the flags.
13. **Everything is a draft until it is reviewed.** The notes of each instrument and the file `docs/dr/PROGRESS.md` list what is known and what is still an open question.

An example of traps 8 and 9: the share of inclusion visits that hold a STAI, per centre, before and since 2018. The centres are shown by their code; a centre with few inclusions in a period gives an unstable share, so look at the counts too.

In [ ]:
period = (inclusion["visit_date"].dt.year >= 2018).map({False: "until 2017", True: "since 2018"})

by_centre = inclusion.groupby(["siteid", period])["staya_total"].agg(["size", lambda column: round(100 * column.notna().mean())])
by_centre.columns = ["inclusions", "with a STAI (%)"]
by_centre.unstack()

And the same score in each centre. A centre with few patients gives an unstable mean, so look at `count` too.

In [ ]:
inclusion.groupby("siteid")["madrs_total"].agg(["count", "mean"]).round(1)

## 13. Question 1: response and remission

How many patients respond, and how many are in remission, six months, one year and two years after the inclusion?

Two usual definitions on the MADRS are used here; both are choices that you can change in the cell:

- **response**: the MADRS fell by half or more since the inclusion visit;
- **remission**: the MADRS is at 10 or less.

**Step 1: one row per patient, one column per visit.** `pivot` does it.

In [ ]:
visits = ["V0", "V6_mois", "V1_an", "V18_mois", "V2_ans"]

madrs = data[data["visit"].isin(visits)].pivot(index="usubjid", columns="visit", values="madrs_total")[visits]

print(madrs.notna().sum())

**Step 2: response and remission at each visit**, among the patients who have a MADRS at inclusion and at that visit.

In [ ]:
REMISSION = 10          # MADRS at or under this value
RESPONSE = 0.5          # share of the inclusion score that must be lost

rows = []
for visit in visits[1:]:
    both = madrs[["V0", visit]].dropna()
    both = both[both["V0"] > 0]
    drop = (both["V0"] - both[visit]) / both["V0"]
    rows.append({"visit": visit, "patients": len(both), "response (%)": round(100 * (drop >= RESPONSE).mean()), "remission (%)": round(100 * (both[visit] <= REMISSION).mean())})

outcome = pd.DataFrame(rows).set_index("visit")
print(outcome)

outcome[["response (%)", "remission (%)"]].plot.bar(title="Response and remission on the MADRS", rot=0)
plt.show()

**Step 3: the same with the patient's own rating.** On the QIDS-SR16, remission is usually a score of 5 or less. The clinician and the patient do not always agree.

In [ ]:
qids = data[data["visit"].isin(visits)].pivot(index="usubjid", columns="visit", values="qids_total")[visits]

rows = []
for visit in visits[1:]:
    both = qids[["V0", visit]].dropna()
    both = both[both["V0"] > 0]
    drop = (both["V0"] - both[visit]) / both["V0"]
    rows.append({"visit": visit, "patients": len(both), "response (%)": round(100 * (drop >= 0.5).mean()), "remission (%)": round(100 * (both[visit] <= 5).mean())})

pd.DataFrame(rows).set_index("visit")

**Step 4: one status per patient at one year.** Each patient gets one status only: a patient in remission is counted as "remission", and "response, no remission" is a patient whose MADRS fell by half without reaching 10. This table has one row per included patient, so it also counts those who were not seen. That is the honest way to present the result: "of 100 patients included, so many are known to be in remission at one year".

In [ ]:
one_year = madrs[["V0", "V1_an"]].copy()
drop = (one_year["V0"] - one_year["V1_an"]) / one_year["V0"]

one_year["status"] = "no response"
one_year.loc[drop >= RESPONSE, "status"] = "response, no remission"
one_year.loc[one_year["V1_an"] <= REMISSION, "status"] = "remission"
one_year.loc[one_year["V1_an"].isna(), "status"] = "not seen at one year"
one_year.loc[one_year["V0"].isna(), "status"] = "no MADRS at inclusion"

print(one_year["status"].value_counts())
print()
print((one_year["status"].value_counts(normalize=True) * 100).round(0))

**Step 5: who was not seen?** Compare, at inclusion, the patients of each status. If the patients who were not seen differ from the others, the percentages of step 2 do not describe the whole cohort.

In [ ]:
with_status = inclusion.merge(one_year[["status"]], left_on="usubjid", right_index=True)

with_status.groupby("status")[["age", "madrs_total", "qids_total", "dsm4_lt_n_mde"]].agg(["count", "mean"]).round(1)

**What this does not tell you.** A patient included less than a year ago cannot have a one-year visit yet: for a rate of follow-up, keep only the patients included long enough ago (`inclusion["visit_date"]`). And a difference between groups here is a description; for a model that takes the centre and the missing visits into account, ask a statistician.

## 14. Question 2: resistance and treatments

How resistant is the depression of these patients, and what did they receive?

**The stage of resistance** of Thase and Rush goes from 1 (one adequate antidepressant failed) to 5 (bilateral electroconvulsive therapy failed too). It is rated at the screening visit.

In [ ]:
stage = data[data["visit"] == "screening"].set_index("usubjid")["thase_rush_stage"]

print(stage.notna().sum(), "patients have a stage")
stage.value_counts().sort_index().plot.bar(title="Stage of resistance (Thase and Rush) at screening", rot=0)
plt.show()

**The history of antidepressant trials (ATHF).** For the current episode, each antidepressant has a rating: 0 means never prescribed, 1 or 2 a trial that was too short or under the adequate dose, 3 or more an adequate trial.

The database stores no score. The cell below counts, for each patient, the antidepressants with an adequate trial. This count is ours, a simple description; it is not a validated score of resistance.

In [ ]:
antidepressants = ["athf_tricyclic_dose", "athf_fluoxetine", "athf_fluvoxamine", "athf_paroxetine", "athf_sertraline", "athf_citalopram", "athf_escitalopram", "athf_duloxetine",
                   "athf_venlafaxine", "athf_milnacipran", "athf_iproniazid", "athf_moclobemide", "athf_toloxatone", "athf_phenelzine", "athf_mianserin", "athf_mirtazapine", "athf_tianeptine",
                   "athf_agomelatine", "athf_bupropion", "athf_vortioxetine"]

filled = inclusion[antidepressants].notna().any(axis=1)
inclusion["adequate_trials"] = (inclusion[antidepressants] >= 3).sum(axis=1).where(filled)

print(filled.sum(), "inclusion visits have the ATHF")
inclusion["adequate_trials"].value_counts().sort_index().plot.bar(title="Antidepressants with an adequate trial, current episode", rot=0)
plt.show()

Which antidepressants were tried, adequately or not? The share of the patients with a rating above 0 and with a rating of 3 or more.

In [ ]:
tried = pd.DataFrame({
    "tried (%)": (inclusion.loc[filled, antidepressants] > 0).mean() * 100,
    "adequate trial (%)": (inclusion.loc[filled, antidepressants] >= 3).mean() * 100,
}).round(0).sort_values("tried (%)")
tried.index = tried.index.str.replace("athf_", "")

tried.plot.barh(figsize=(9, 6), title="Antidepressants of the current episode (ATHF)")
plt.show()

Does the count agree with the stage? The stage is on the screening row, so we bring it to the inclusion table with `merge`. The inclusion table has a column of the same name, empty: we drop it first. More adequate trials should go with a higher stage.

In [ ]:
inclusion = inclusion.drop(columns="thase_rush_stage").merge(stage.reset_index(), on="usubjid", how="left")

inclusion.groupby("thase_rush_stage")[["adequate_trials", "madrs_total"]].agg(["count", "mean"]).round(1)

**Other treatments of the current episode.** The ATHF also rates electroconvulsive therapy, magnetic stimulation and the augmentation agents. And the inclusion form asks whether the patient ever had electroconvulsive therapy, magnetic stimulation or a psychotherapy.

In [ ]:
others = ["athf_ect_unilateral", "athf_ect_bilateral", "athf_tms", "athf_lithium_augmentation", "athf_lamotrigine", "athf_thyroid_hormones", "athf_cbt"]
print(((inclusion.loc[filled, others] > 0).mean() * 100).round(0))
print()

ever = inclusion[["ever_ect", "ever_tms", "ever_psychotherapy"]].agg(["count", "mean"]).T
ever["percent"] = (ever["mean"] * 100).round(0)
print(ever[["count", "percent"]])

**The current treatments, without the dates.** The list of current medications says which products a patient has, even when the doses and dates are lost (section 8). So we can count the patients who have each group, and the combinations.

`pd.crosstab` makes one row per patient and one column per group; `> 0` turns the counts into yes or no.

In [ ]:
current = treatments[treatments["list"] == "medications"]
has = pd.crosstab(current["usubjid"], current["treatment_group"]) > 0

print(len(has), "patients have a list of current psychotropic treatments")
print()
print((has.mean() * 100).round(0).sort_values(ascending=False))
print()
print("Antidepressant and lithium:", (has["antidepressant"] & has["lithium"]).sum(), "patients")
print("Antidepressant and second-generation antipsychotic:", (has["antidepressant"] & has["antipsychotic_2g"]).sum(), "patients")
print("Two antidepressants or more:", (current[current["treatment_group"] == "antidepressant"].groupby("usubjid")["molecule"].nunique() >= 2).sum(), "patients")

**Electroconvulsive therapy during the follow-up.** At the follow-up visits the question is "since the last visit" and has its own column, `ect_fu` (trap 2).

In [ ]:
data[data["visit"].isin(["V6_mois", "V1_an", "V18_mois", "V2_ans", "V4_ans"])].groupby("visit_month")["ect_fu"].agg(["count", "mean"]).round(2)

## 15. Question 3: MINI comorbidities

Which psychiatric disorders come with the depression? The MINI, a structured diagnostic interview, is given at the screening visit.

**Step 1: put the screening and the inclusion of a patient side by side.** We take the MINI columns from the screening rows and join them to the inclusion rows. The inclusion rows have the same columns, empty; we drop them first so that the join does not create doubles.

In [ ]:
mini_columns = ["mini_a01a", "mini_suicide_risk", "mini_gad_current", "mini_social_phobia_current", "mini_panic_lifetime", "mini_agoraphobia_without_panic_current", "mini_ptsd_current",
                "mini_alcohol_dependence_current"]

from_screening = data[data["visit"] == "screening"][["usubjid"] + mini_columns]
first = data[data["visit"] == "V0"].drop(columns=mini_columns).merge(from_screening, on="usubjid", how="left")

print(len(first), "patients;", first["mini_a01a"].notna().sum(), "with an answered MINI")

**Step 2: how frequent is each disorder?** We keep the patients whose MINI was answered (the first question of the interview, `mini_a01a`, holds an answer), because the conclusion boxes hold 0 when the form was only opened (trap 4).

In [ ]:
answered = first[first["mini_a01a"].notna()]
conclusions = mini_columns[1:]

table = answered[conclusions].agg(["count", "mean"]).T
table["percent"] = (table["mean"] * 100).round(0)
table[["count", "percent"]].sort_values("percent", ascending=False)

**Step 3: are patients with an anxiety disorder more severe at inclusion?** A patient has "an anxiety disorder" here when at least one of four conclusions is yes. `max(axis=1)` gives 1 when at least one column is 1.

In [ ]:
anxiety = ["mini_gad_current", "mini_social_phobia_current", "mini_panic_lifetime", "mini_agoraphobia_without_panic_current"]
answered = answered.assign(anxiety_disorder=answered[anxiety].max(axis=1))

print(answered.groupby("anxiety_disorder")[["madrs_total", "qids_total", "staya_total", "gaf"]].agg(["count", "mean"]).round(1).T)

answered.boxplot(column="qids_total", by="anxiety_disorder")
plt.show()

**Step 4: the number of comorbid disorders.** Count the conclusions that are yes, patient by patient.

In [ ]:
disorders = ["mini_gad_current", "mini_social_phobia_current", "mini_panic_lifetime", "mini_agoraphobia_without_panic_current", "mini_ptsd_current", "mini_alcohol_dependence_current"]

answered[disorders].sum(axis=1).value_counts().sort_index().plot.bar(title="Number of comorbid disorders at screening (of six)", rot=0)
plt.show()

**Two cautions.** The MINI is answered for about three patients in four, and less often since 2021: say how many patients your percentages rest on. And the MINI is given again at two and four years to some patients; those rows are follow-up visits, not a second screening.

## 16. Question 4: life events and trauma

Do childhood trauma and life events go with a more severe depression?

Four instruments describe them:

- the **CTQ** (childhood trauma questionnaire): a total from 25 to 125 and five sub-scores, at inclusion;
- the **traumatic events** list: `trauma_events`, the number of events of a list of 17;
- the **Paykel** list of recent life events: `paykel_events`;
- the **Amiel-Lebigre** list of life events: `amiel_events`.

The CTQ scores are the corrected ones (section 5).

In [ ]:
trauma = ["ctq_total", "ctq_emotional_abuse", "ctq_physical_abuse", "ctq_sexual_abuse", "ctq_emotional_neglect", "ctq_physical_neglect", "trauma_events", "paykel_events", "amiel_events"]

print(inclusion[trauma].describe().round(1).T[["count", "mean", "50%", "max"]])

inclusion["ctq_total"].plot.hist(bins=30, title="Childhood trauma questionnaire, total at inclusion")
plt.show()

**Against the severity at inclusion.** A table of correlations: 0 means no link, 1 a perfect one. Then the picture for one pair.

In [ ]:
severity = ["madrs_total", "qids_total", "shaps_total", "dsm4_first_episode_age", "dsm4_lt_n_mde"]

print(inclusion[["ctq_total", "trauma_events", "paykel_events"] + severity].corr().loc[severity, ["ctq_total", "trauma_events", "paykel_events"]].round(2))

inclusion.plot.scatter(x="ctq_total", y="qids_total", alpha=0.3, title="Childhood trauma (CTQ) against depression rated by the patient (QIDS)")
plt.show()

**By quarter of the CTQ.** `pd.qcut` cuts the patients into four groups of the same size, from the lowest to the highest total. The suicide attempt is a yes/no column, so its mean is a share.

In [ ]:
inclusion["ctq_quarter"] = pd.qcut(inclusion["ctq_total"], 4, labels=["lowest", "second", "third", "highest"])

inclusion.groupby("ctq_quarter", observed=True)[["ctq_total", "madrs_total", "qids_total", "dsm4_first_episode_age", "cssrs_attempt_lifetime"]].mean().round(2)

**And the outcome at one year?** We join the status of question 1 (section 13) and compare the CTQ of each group. Remember who is missing: the patients not seen at one year are a group of their own.

In [ ]:
with_status = inclusion.merge(one_year[["status"]], left_on="usubjid", right_index=True)

with_status.groupby("status")[["ctq_total", "trauma_events"]].agg(["count", "mean"]).round(1)

**Two cautions.** The Paykel list is filled by about one inclusion in four since 2018 (trap 8): look at `count`. And the wording of the events of the Paykel and trauma lists is not in the dictionary, so their items are numbered (`paykel01`, `trau01`...) and only the counts can be used today. The 53 Amiel-Lebigre events are numbered too (`amiel01`...) but their labels give the wording of the form; the list is another one at follow-up (`amiel_events_fu`).

## 17. Build your own table and save it

A typical analysis needs one row per patient with a handful of columns. Choose the visit, choose the columns, join the patient columns, and save.

The table `inclusion` already holds what the sections above added: the sex, the stage of resistance, the number of adequate trials. The file goes to `data/exports/`, which stays on this computer.

In [ ]:
from pathlib import Path

columns = ["usubjid", "sex", "visit_date", "siteid", "age", "madrs_total", "qids_total", "cgi_severity", "gaf", "ctq_total", "bmi", "thase_rush_stage", "adequate_trials"]

my_table = inclusion[columns].merge(one_year[["status"]], left_on="usubjid", right_index=True)

Path("data/exports").mkdir(exist_ok=True)
my_table.to_csv("data/exports/my_first_dr_table.csv", index=False)

print(len(my_table), "patients and", my_table.shape[1], "columns saved")

A first question with this table: is a higher stage of resistance at screening followed by fewer remissions at one year?

This is a description, not a proof. For a test or a model that takes age, sex and centre into account, open the saved file in your statistics software (R, SPSS, Jamovi) or ask a statistician.

In [ ]:
seen = my_table[my_table["status"].isin(["remission", "response, no remission", "no response"])]

print(pd.crosstab(seen["thase_rush_stage"], seen["status"]))
print()
print((pd.crosstab(seen["thase_rush_stage"], seen["status"], normalize="index") * 100).round(0))

## 18. Load only what you need

Loading everything takes half a minute. When you know which instruments you need, name them: it takes a few seconds. The names are in the column `instrument` of the catalogue.

In [ ]:
small = dr.load(EXTRACTION, instruments=["madrs", "qids_sr16", "cgi_egf"])

print(small.data.shape[1], "columns instead of", data.shape[1])

## 19. Put the bipolar, schizophrenia and depression cohorts together

The three cohorts are loaded by the same package, and the instruments that several cohorts have keep the same names and the same columns. So their tables can be stacked one under the other.

**Step 1: which instruments do the cohorts share?** The cell below compares the lists of instruments. It reads no patient data.

In [ ]:
from facedata import bp, sz

BP_EXTRACTION = "data/bp/BPALL_20261113_2.tsv"     # the bipolar file you received
SZ_EXTRACTION = "data/sz/SZALL_20261117.tsv"       # the schizophrenia file you received

names = {"bp": set(load_schemas(bp.SCHEMAS)), "sz": set(load_schemas(sz.SCHEMAS)), "dr": set(load_schemas(dr.SCHEMAS))}

in_all_three = sorted(names["bp"] & names["sz"] & names["dr"])
print(len(in_all_three), "instruments have the same name in the three cohorts:")
print(", ".join(in_all_three))
print()
print(len(names["dr"] & names["bp"]), "are shared by depression and bipolar;", len(names["dr"] & names["sz"]), "by depression and schizophrenia")

**Step 2: load the same instruments from the three files.** We take the identity, the CGI and the GAF, the YMRS, the childhood trauma questionnaire, weight and height, tobacco and the medications, which the three cohorts have, and the MADRS, which the bipolar and the depression cohorts have.

A cohort is asked only for the instruments it has.

In [ ]:
wanted = ["identite", "cgi_egf", "ymrs", "ctq", "parametres_physiques", "tobacco", "medications", "madrs"]

bipolar = bp.load(BP_EXTRACTION, instruments=[name for name in wanted if name in names["bp"]])
schizophrenia = sz.load(SZ_EXTRACTION, instruments=[name for name in wanted if name in names["sz"]])
depression = dr.load(EXTRACTION, instruments=[name for name in wanted if name in names["dr"]])

print("bipolar:", bipolar.data.shape, " schizophrenia:", schizophrenia.data.shape, " depression:", depression.data.shape)

**Step 3: stack the tables** with `pd.concat`. The first column, `cohort`, says where each row comes from. A column that a cohort does not have is empty for its rows.

In [ ]:
three = pd.concat([bipolar.data, schizophrenia.data, depression.data], ignore_index=True)
three_patients = pd.concat([bipolar.patients, schizophrenia.patients, depression.patients], ignore_index=True)

print(three["cohort"].value_counts())
print()
print(three_patients.groupby("cohort")["sex"].value_counts())

**The key of a patient is `cohort` with `usubjid`.** A patient number is given inside a cohort: the same number can be another person in another cohort. So every `merge` and every `groupby` of the stacked tables must name both columns.

`patients` also holds `fondacode`, a patient code that is unique across the cohorts and does not change from one extraction to the next. The cell below counts the numbers of the depression cohort that are also used in another cohort, and checks that no `fondacode` appears twice.

In [ ]:
numbers = {cohort: set(table["usubjid"]) for cohort, table in three_patients.groupby("cohort")}

print(len(numbers["dr"] & numbers["bp"]), "numbers of the depression cohort are also bipolar numbers, for other patients")
print(len(numbers["dr"] & numbers["sz"]), "are also schizophrenia numbers")
print(three_patients["fondacode"].duplicated().sum(), "fondacode appears twice")

**Step 4: join and compare.** The inclusion visit of every patient, with the sex. Note the two columns in `on=`.

In [ ]:
first_three = three[three["visit"] == "V0"].merge(three_patients[["cohort", "usubjid", "sex"]], on=["cohort", "usubjid"])

first_three.groupby("cohort")[["age", "bmi", "gaf", "cgi_severity", "ymrs_total", "ctq_total", "madrs_total"]].agg(["count", "mean"]).round(1).T

In [ ]:
first_three.boxplot(column="gaf", by="cohort", figsize=(8, 4))
plt.title("Global functioning (GAF) at inclusion, by cohort")
plt.suptitle("")
plt.show()

Childhood trauma in the three cohorts: three histograms on the same axes. `density=True` makes them comparable although the cohorts do not have the same size. The comparison makes sense because the depression answers were brought back to the coding of the two other cohorts.

In [ ]:
for name, group in first_three.groupby("cohort"):
    group["ctq_total"].plot.hist(bins=30, alpha=0.5, density=True, label=name)

plt.title("Childhood trauma questionnaire, total score at inclusion")
plt.legend()
plt.show()

The MADRS exists in the bipolar and the depression cohorts. Bipolar patients are included in any mood state, depressed patients during a resistant episode: the two distributions tell it.

In [ ]:
for name, group in first_three[first_three["cohort"].isin(["bp", "dr"])].groupby("cohort"):
    group["madrs_total"].plot.hist(bins=30, alpha=0.5, density=True, label=name)

plt.title("MADRS at inclusion")
plt.legend()
plt.show()

Tobacco, as shares of each cohort. `pd.crosstab` counts the patients for each pair of values; `normalize="index"` turns each row into shares.

In [ ]:
(pd.crosstab(first_three["cohort"], first_three["tobacco_status"], normalize="index") * 100).round(0)

**Over time, in the three cohorts.** The same move as in section 10, with `cohort` added to the `groupby`. We keep the visits that the three cohorts have: inclusion, one year and two years.

In [ ]:
common_visits = three[three["visit"].isin(["V0", "V1_an", "V2_ans"])]

print(common_visits.groupby(["visit_month", "cohort"])["gaf"].count().unstack())

common_visits.groupby(["visit_month", "cohort"])["gaf"].mean().unstack().plot(marker="o", title="Mean GAF at inclusion, one year and two years")
plt.xlabel("Months since inclusion")
plt.show()

**The medications stack too.** One row per patient and medication. The depression table has one more column, `list` (section 8); it is empty for the two other cohorts, whose table holds one list only.

In [ ]:
three_treatments = pd.concat([bipolar.treatments, schizophrenia.treatments, depression.treatments], ignore_index=True)

top = three_treatments["molecule"].value_counts().head(12).index
pd.crosstab(three_treatments["molecule"], three_treatments["cohort"]).loc[top]

**Three things to check before you compare the cohorts.**

1. **A column that only one cohort has is empty for the others.** The MADRS is not in the schizophrenia cohort. Count the values per cohort before you use a column.
2. **The same name does not always mean exactly the same form.** The file `facedata/dr/differences.yaml` lists, for each shared instrument, what differs in the depression cohort. The cell below prints it for the instruments we loaded.
3. **The cohorts were not recruited the same way**, and their centres and years differ. A difference between cohorts is a description, not an effect of the diagnosis.

In [ ]:
import yaml

print(three.groupby("cohort")[["gaf", "ctq_total", "madrs_total", "ymrs_total"]].count())
print()

differences = yaml.load(open("facedata/dr/differences.yaml", encoding="utf-8"), Loader=yaml.BaseLoader)
for name in wanted:
    print(name, ":", differences.get(name, "the same as in the cohort it is shared with"))

**Save the stacked table**, as in section 17. To stack everything, load the three cohorts without `instruments=` and use the same `pd.concat`: it takes a few minutes and gives a table of several thousand columns, most of them empty for two cohorts.

In [ ]:
first_three.to_csv("data/exports/three_cohorts_inclusion.csv", index=False)

print(len(first_three), "patients of", first_three["cohort"].nunique(), "cohorts and", first_three.shape[1], "columns saved")

## Where to go next

- The notes of each instrument (section 4) and the report (section 11).
- `README.md` for the package, and `docs/dr/PROGRESS.md` for every decision taken about the data and every open question.
- `bipolar-tutorial.ipynb` and `schizophrenia-tutorial.ipynb`, the same tour for the two other cohorts.
- The five moves used in this notebook cover most needs: keep rows with `table[table["column"] == value]`, count with `value_counts()`, summarise with `groupby(...).mean()`, join with `merge`, and draw with `.plot`.

**Before you close: clear the results** (menu *Edit*, *Clear Outputs of All Cells*) if this notebook will be shared.